In [ ]:
using Pkg 
Pkg.activate("..")

In [ ]:
include("../src/PhasorNetworks.jl")
using .PhasorNetworks, Plots, DifferentialEquations

In [ ]:
using Lux, MLUtils, OneHotArrays, Statistics, Test, LuxCUDA
using Random: Xoshiro, AbstractRNG
using Base: @kwdef
using Zygote: withgradient
using Optimisers, ComponentArrays
using Statistics: mean
using LinearAlgebra: diag
using PhasorNetworks: bind
using Distributions: Normal
using DifferentialEquations: Heun, Tsit5

In [ ]:
solver_args = Dict(:adaptive => false, :dt => 0.01f0)

In [ ]:
spk_args = SpikingArgs(threshold = 0.001f0,
                    solver=Tsit5(), 
                    solver_args = solver_args)

In [ ]:
cdev = cpu_device()
gdev = gpu_device()

In [ ]:
args = Args(batchsize = 128, epochs = 20, use_cuda = true)

In [ ]:
rng = Xoshiro(42)

In [ ]:
phases = random_symbols(rng, (128, 128, 16, 32));

In [ ]:
spikes = phase_to_train(phases, spk_args=spk_args, repeats=5)

In [ ]:
layer = Chain(MaxPool((2,2)),)

In [ ]:
ps, st = Lux.setup(rng, layer)

In [ ]:
op_a, _ = layer(phases, ps, st)

In [ ]:
op_a |> size

In [ ]:
import .PhasorNetworks: LuxParams

In [ ]:
scall = SpikingCall(spikes, spk_args, (0.0f0, 10.0f0))

In [ ]:
pval = train_to_phase(scall)

In [ ]:
import Lux: MaxPool

In [ ]:
import .PhasorNetworks: generate_cycles

In [ ]:
generate_cycles(scall.t_span, scall.spk_args, scall.train.offset)

In [ ]:
import .PhasorNetworks: vcat_trains, SpikingTypes, on_gpu

In [ ]:
scall

In [ ]:
41943040 / 12582919

In [ ]:
mvs, cyc = layer(scall, ps, st)

In [ ]:
mvs

In [ ]:
out_vals = train_to_phase(mvs);

In [ ]:
size(out_vals)

In [ ]:
phases |> size

In [ ]:
m_phases, _ = layer(phases, ps, st);

In [ ]:
size(m_phases)

In [ ]:
scatter(vec(m_phases[:,:,:,1]), vec(out_vals[:,:,:,1,2]))